# 马尔可夫决策过程

## 前置保留部分

In [1]:
# 导包
import numpy as np

# 环境与策略定义
S = ["S1","S2","S3","S4","S5"]
A = ["保持S1","前往S1","前往S2","前往S3","前往S4","前往S5","概率前往"]

# 状态转移函数
P = {
    "S1-保持S1-S1":1.0, "S1-前往S2-S2":1.0,
    "S2-前往S1-S1":1.0, "S2-前往S3-S3":1.0,
    "S3-前往S4-S4":1.0, "S3-前往S5-S5":1.0,
    "S4-前往S5-S5":1.0, "S4-概率前往-S2":0.2,
    "S4-概率前往-S3":0.4, "S4-概率前往-S4":0.4,
}
# 奖励函数
R = {
    "S1-保持S1":-1, "S1-前往S2":0,
    "S2-前往S1":-1, "S2-前往S3":-2,
    "S3-前往S4":-2, "S3-前往S5":0,
    "S4-前往S5":10, "S4-概率前往":1,
}
gamma = 0.5
MDP = (S, A, P, R, gamma)

# 策略1，随机策略
Pi_1 = {
    "S1-保持S1":0.5, "S1-前往S2":0.5,
    "S2-前往S1":0.5, "S2-前往S3":0.5,
    "S3-前往S4":0.5, "S3-前往S5":0.5,
    "S4-前往S5":0.5, "S4-概率前往":0.5,
}
# 策略2
Pi_2 = {
    "S1-保持S1":0.6, "S1-前往S2":0.4,
    "S2-前往S1":0.3, "S2-前往S3":0.7,
    "S3-前往S4":0.5, "S3-前往S5":0.5,
    "S4-前往S5":0.1, "S4-概率前往":0.9,
}

def join(str1, str2):
    return str1 + '-' + str2

# 采样函数
def sample(MDP, Pi, timestep_max, number):
    S, A, P, R, gamma = MDP
    episodes = []
    for _ in range(number):
        episode = []
        timestep = 0
        s = S[np.random.randint(4)] # 随机选起点（除了S5）
        while s != "S5" and timestep <= timestep_max:
            timestep += 1
            rand, temp = np.random.rand(), 0
            # 选动作
            for a_opt in A:
                temp += Pi.get(join(s, a_opt), 0)
                if temp > rand:
                    a = a_opt
                    r = R.get(join(s, a), 0)
                    break
            rand, temp = np.random.rand(), 0
            # 选下一个状态
            for s_opt in S:
                temp += P.get(join(join(s, a), s_opt), 0)
                if temp > rand:
                    s_next = s_opt
                    break
            episode.append((s, a, r, s_next))
            s = s_next
        episodes.append(episode)
    return episodes

## 占用度量（Occupancy Measure）
接下来我们用代码来近似估计占用度量函数。这里我们采用近似估计，设置一个较大的episode length最大值，然后采样很多次，用(s, a)组出现的频率估计实际概率。

In [4]:
# 参1：轨迹组；参2：目标状态；参3：目标动作；参4：最长时间步；参5：折扣因子
def occupancy(episodes, s, a, timestep_max, gamma):
    """计算状态-动作对(s, a)出现的概率。以此来估算策略的占用度量"""
    rho = 0
    total_times = np.zeros(timestep_max)        # 分母数组：记录每个时间步t各被经历过几次
    occur_times = np.zeros(timestep_max)        # 分子数组：记录(s_t, a_t) = (s, a)的次数

    for episode in episodes:                    # 遍历每条轨迹
        for i in range(len(episode)):           # 遍历单条轨迹的每一步
            (s_opt, a_opt, r, s_next) = episode[i]      # 获取第 i 步的信息，包括状态、动作、奖励、下一状态
            total_times[i] += 1     # 证明走到了第 i 步，第 i 步的总数 +1
            if s == s_opt and a == a_opt:       # 正好遇到目标状态-动作对，分子 +1
                occur_times[i] += 1

    for i in reversed(range(timestep_max)):         # 计算时间折扣累加
        if total_times[i]:      # 如果第i步没有轨迹到达，就跳过
            # gamma**i 是时间折扣；occur_times[i] / total_times[i] 是条件概率 P_t(s,a)
            rho += gamma ** i * occur_times[i] / total_times[i]
    return (1 - gamma) * rho        # 返回归一化后的占用度量

In [6]:
gamma = 0.5
timestep_max = 1000

episodes_1 = sample(MDP, Pi_1, timestep_max, 1000)      # 通过策略1生成轨迹1
episodes_2 = sample(MDP, Pi_2, timestep_max, 1000)      # 通过策略2生成轨迹2
rho_1 = occupancy(episodes_1, "S4", "概率前往", timestep_max, gamma)    # 计算轨迹1中 ("S4", "概率前往")这一状态-动作对的占用度量
rho_2 = occupancy(episodes_2, "S4", "概率前往", timestep_max, gamma)    # 计算轨迹2中 ("S4", "概率前往")这一状态-动作对的占用度量
print(rho_1, rho_2)

0.10885755059049404 0.24061655758454342


我们可以发现不同策略对于同一个状态动作对 (s, a) 的占用度量是不一样的。

## 总结
在本章内容中，我们从零开始学习了马尔可夫决策过程的一些基础概念知识，以及如何求解贝尔曼方程得到状态价值的解析解和用蒙特卡洛方法估计各个状态的价值。马尔可夫决策过程是强化学习的基础，之后学习的强化学习方法通常都是在求解马尔可夫决策过程中的最优策略。即强化学习中的环境就是一个马尔可夫决策过程。